# 实验二：心血管既往史数据清洗与SVM/决策树建模

本Notebook完成：数据清洗、特征工程，使用SVM与决策树完成分类建模与评估，并自动生成Word实验报告与保存Jupyter代码。

In [5]:
# 1) 配置与库导入（可在左侧“运行所有”前先执行本单元）
import sys, os, warnings, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 机器学习与评估
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
)

# 重要性与持久化
from sklearn.inspection import permutation_importance
import joblib

# 生成Word报告
from docx import Document
from docx.shared import Inches

warnings.filterwarnings('ignore')
np.random.seed(42)

# Matplotlib中文与样式
plt.style.use('seaborn-v0_8')
plt.rcParams['axes.unicode_minus'] = False
# Windows 常用中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('sklearn:', __import__('sklearn').__version__)


Python: 3.12.10
pandas: 2.3.3
sklearn: 1.7.2


In [7]:
# 3) 初步探索（行列数、缺失、唯一值）
print('行/列:', raw_main.shape)
print('\n基本信息:')
print(raw_main.info())

print('\n描述统计:')
display(raw_main.describe(include='all').T.head(20))

print('\n每列缺失率:')
na_rate = raw_main.replace(['nan',' NAN',' Nan','NaN','NA',''], np.nan).isna().mean().sort_values(ascending=False)
display(na_rate.head(20))

# 目标列分布（预计为“既往史-心血管”）
if '既往史-心血管' in raw_main.columns:
    print('\n目标列分布:')
    display(raw_main['既往史-心血管'].value_counts(dropna=False))
else:
    print('[警告] 未找到目标列 `既往史-心血管`，后续将动态检查。')


行/列: (1000, 24)

基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 24 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   健康档案编号    1000 non-null   int64  
 1   性别        1000 non-null   object 
 2   年份        1000 non-null   int64  
 3   年龄        1000 non-null   float64
 4   腰围        1000 non-null   float64
 5   体质指数      1000 non-null   float64
 6   锻炼频率      1000 non-null   object 
 7   吸烟状况      1000 non-null   object 
 8   饮食习惯      1000 non-null   object 
 9   饮酒频率      1000 non-null   object 
 10  空腹血糖MMOL  1000 non-null   float64
 11  血清谷丙转氨酶   1000 non-null   float64
 12  血清谷草转氨酶   1000 non-null   float64
 13  总胆固醇      1000 non-null   float64
 14  甘油三酯      1000 non-null   float64
 15  收缩压       1000 non-null   float64
 16  舒张压       1000 non-null   float64
 17  心电图合并     1000 non-null   object 
 18  健康评价合并    1000 non-null   object 
 19  疾病描述      1000 non-null   object 
 20  既往史-高血压 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
健康档案编号,1000.0,NaN,NaN,NaN,65311533122741712.0,13548713945975.982422,65010201601103272.0,65312200401100632.0,65312200802200472.0,65312201200825408.0,65312901000501864.0
性别,1000,2,女,600,NaN,NaN,NaN,NaN,NaN,NaN,NaN
年份,1000.0,NaN,NaN,NaN,2018.0,0.0,2018.0,2018.0,2018.0,2018.0,2018.0
年龄,1000.0,NaN,NaN,NaN,54.262,12.779385,36.0,44.0,53.0,64.0,103.0
腰围,1000.0,NaN,NaN,NaN,87.236,12.605582,48.0,79.0,86.0,96.0,144.0
体质指数,1000.0,NaN,NaN,NaN,25.01997,4.712613,12.49,21.615,24.715,27.74,51.06
锻炼频率,1000,4,不锻炼,974,NaN,NaN,NaN,NaN,NaN,NaN,NaN
吸烟状况,1000,3,从不吸烟,868,NaN,NaN,NaN,NaN,NaN,NaN,NaN
饮食习惯,1000,5,荤素均衡,757,NaN,NaN,NaN,NaN,NaN,NaN,NaN
饮酒频率,1000,3,从不,917,NaN,NaN,NaN,NaN,NaN,NaN,NaN



每列缺失率:


健康档案编号      0.0
性别          0.0
年份          0.0
年龄          0.0
腰围          0.0
体质指数        0.0
锻炼频率        0.0
吸烟状况        0.0
饮食习惯        0.0
饮酒频率        0.0
空腹血糖MMOL    0.0
血清谷丙转氨酶     0.0
血清谷草转氨酶     0.0
总胆固醇        0.0
甘油三酯        0.0
收缩压         0.0
舒张压         0.0
心电图合并       0.0
健康评价合并      0.0
疾病描述        0.0
dtype: float64


目标列分布:


既往史-心血管
无    700
有    300
Name: count, dtype: int64

In [8]:
# 4) 统一列名与数据类型 + 5) 清洗异常值与噪声文本

def normalize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    def norm(c):
        c = str(c).strip()
        # 半角/全角与奇异字符简化
        trans = {
            '（':'(', '）':')', '：':':', '，':',', '。':'.', '、':'/', '　':' ', '\t':' ', '\n':' ', '\r':' '
        }
        for k,v in trans.items():
            c = c.replace(k,v)
        return c
    df = df.copy()
    df.columns = [norm(c) for c in df.columns]
    return df

main = normalize_colnames(raw_main)
if raw_ref is not None:
    ref  = normalize_colnames(raw_ref)
else:
    ref = None

# 替换常见字符串缺失
main = main.replace({r'^\s*$':'NaN', r'^(nan|NaN|NAN|None|null)$':'NaN'}, regex=True)
main = main.replace('NaN', np.nan)

# 关键数值列列表（存在则进行数值转换）
num_candidates = [
    '年龄','腰围','体质指数','空腹血糖MMOL','血清谷丙转氨酶','血清谷草转氨酶','总胆固醇','甘油三酯','收缩压','舒张压'
]
for col in num_candidates:
    if col in main.columns:
        main[col] = pd.to_numeric(main[col], errors='coerce')

# 地区列
if '北方地区' in main.columns:
    main['北方地区'] = pd.to_numeric(main['北方地区'], errors='coerce').fillna(0).astype(int)

# 6) 缺失值处理策略将在Pipeline中完成（Imputer）

print('清洗后基本形状:', main.shape)
main.head()

清洗后基本形状: (1000, 24)


,健康档案编号,性别,年份,年龄,腰围,体质指数,锻炼频率,吸烟状况,饮食习惯,饮酒频率,...,甘油三酯,收缩压,舒张压,心电图合并,健康评价合并,疾病描述,既往史-高血压,既往史-糖尿病,北方地区,既往史-心血管
0,65312201000100840,女,2018,55.0,94.0,25.96,不锻炼,从不吸烟,素食为主,从不,...,1.80,129.0,83.0,窦性心律及异常,有异常 慢支炎 超重 高血压 nan nan nan,短暂性脑缺血发作 短暂性脑缺血发作 未发现 未发现 未发现 未发现 未发现 nan,无,无,1,有
1,65312201200700953,男,2018,66.0,96.0,27.22,不锻炼,吸烟,荤素均衡,经常,...,3.16,110.0,80.0,心肌缺血/损伤/梗死征象,有异常 心肌缺血 支气管炎 超重 nan nan nan,未发现 未发现 其他 心肌缺血 未发现 未发现 有 支气管炎,无,无,1,有
2,65312200400600172,女,2018,62.0,84.0,21.79,不锻炼,从不吸烟,荤素均衡,从不,...,0.65,130.0,70.0,传导系统障碍,有异常 冠心病 胆囊炎 nan 双肺结核 nan nan,未发现 未发现 心绞痛 心绞痛 未发现 未发现 有 双肺结核，胆囊炎,无,无,1,有
3,65312200400700884,女,2018,55.0,107.0,31.98,不锻炼,从不吸烟,荤素均衡,从不,...,1.60,120.0,70.0,窦性心律及异常,有异常 肥胖 胆囊炎 高血压病 冠心病 nan nan,未发现 未发现 nan 心绞痛#高血压 未发现 未发现 有 胆囊炎,有,无,1,有
4,65312200900401724,女,2018,60.0,73.0,21.78,不锻炼,从不吸烟,荤素均衡,从不,...,0.49,120.0,90.0,窦性心律及异常,体检无异常 nan nan nan nan nan nan,未发现 未发现 冠状动脉血运重建 冠状动脉血运重建 未发现 未发现 未发现 nan,无,无,1,有


In [9]:
# 7) 特征工程（类别编码与数值缩放）+ 8) 构建目标变量

# 选取特征列（存在则使用）
cat_candidates = ['性别','锻炼频率','吸烟状况','饮食习惯','饮酒频率']
text_long_cols = ['心电图合并','健康评价合并','疾病描述']

feature_cols = []
for col in num_candidates + cat_candidates + ['北方地区', '既往史-高血压','既往史-糖尿病']:
    if col in main.columns:
        feature_cols.append(col)

# 删除超长描述列，避免高维文本噪声
use_df = main.drop(columns=[c for c in text_long_cols if c in main.columns], errors='ignore')

# 目标变量映射
if '既往史-心血管' not in use_df.columns:
    raise ValueError('未找到目标列 `既往史-心血管`，请检查数据列名。')

use_df['y'] = use_df['既往史-心血管'].map({'有':1, '无':0}).astype('Int64')

# 历史疾病列映射
for hx in ['既往史-高血压','既往史-糖尿病']:
    if hx in use_df.columns:
        use_df[hx] = use_df[hx].map({'有':1, '无':0}).astype('float')

# 拆分X/y
X = use_df[feature_cols].copy()
y = use_df['y'].astype(int)

# 数值/类别列拆分
num_features = [c for c in num_candidates + ['北方地区','既往史-高血压','既往史-糖尿病'] if c in X.columns]
cat_features = [c for c in cat_candidates if c in X.columns]

print('数值特征:', num_features)
print('类别特征:', cat_features)
print('目标分布:\n', y.value_counts(normalize=True))

数值特征: ['年龄', '腰围', '体质指数', '空腹血糖MMOL', '血清谷丙转氨酶', '血清谷草转氨酶', '总胆固醇', '甘油三酯', '收缩压', '舒张压', '北方地区', '既往史-高血压', '既往史-糖尿病']
类别特征: ['性别', '锻炼频率', '吸烟状况', '饮食习惯', '饮酒频率']
目标分布:
 y
0    0.7
1    0.3
Name: proportion, dtype: float64


In [10]:
# 9) 训练/验证/测试集划分（分层抽样）
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print('Train:', X_train.shape, 'Valid:', X_valid.shape, 'Test:', X_test.shape)


Train: (700, 18) Valid: (150, 18) Test: (150, 18)


In [11]:
# 10) 预处理器（Imputer+OneHot+Scaler）与SVM网格搜索
from sklearn.utils.class_weight import compute_class_weight

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())  # SVM分支需要缩放
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor_for_svm = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

svm_pipe = Pipeline(steps=[
    ('pre', preprocessor_for_svm),
    ('clf', SVC(probability=True, class_weight='balanced', random_state=42))
])

param_grid_svm = {
    'clf__kernel': ['rbf', 'linear'],
    'clf__C': [0.5, 1, 2, 5],
    'clf__gamma': ['scale', 0.1, 0.01]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_gs = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid_svm,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

svm_gs.fit(X_train, y_train)
print('SVM最佳参数:', svm_gs.best_params_)
print('SVM最佳CV AUC:', svm_gs.best_score_)


Fitting 5 folds for each of 24 candidates, totalling 120 fits
SVM最佳参数: {'clf__C': 0.5, 'clf__gamma': 0.01, 'clf__kernel': 'rbf'}
SVM最佳CV AUC: 0.7936831875607386


In [12]:
# 11) 决策树Pipeline+网格搜索（无需数值缩放）
num_transformer_dt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor_for_dt = ColumnTransformer(
    transformers=[
        ('num', num_transformer_dt, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

dt_pipe = Pipeline(steps=[
    ('pre', preprocessor_for_dt),
    ('clf', DecisionTreeClassifier(random_state=42, class_weight='balanced'))
])

param_grid_dt = {
    'clf__max_depth': [None, 4, 6, 8, 10],
    'clf__min_samples_split': [2, 5, 10, 20],
    'clf__min_samples_leaf': [1, 2, 5, 10]
}

dt_gs = GridSearchCV(
    estimator=dt_pipe,
    param_grid=param_grid_dt,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

dt_gs.fit(X_train, y_train)
print('DT最佳参数:', dt_gs.best_params_)
print('DT最佳CV AUC:', dt_gs.best_score_)


Fitting 5 folds for each of 80 candidates, totalling 400 fits
DT最佳参数: {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__min_samples_split': 20}
DT最佳CV AUC: 0.7715986394557823


In [13]:
# 12) 评估与可视化工具函数
from pathlib import Path

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)


def eval_and_plots(name, model, Xv, yv, Xt, yt):
    from sklearn.metrics import auc
    # 验证集
    pv = model.predict(Xv)
    sv = model.predict_proba(Xv)[:,1]
    acc_v = accuracy_score(yv, pv)
    pre_v = precision_score(yv, pv, zero_division=0)
    rec_v = recall_score(yv, pv)
    f1_v  = f1_score(yv, pv)
    auc_v = roc_auc_score(yv, sv)

    # 测试集
    pt = model.predict(Xt)
    st = model.predict_proba(Xt)[:,1]
    acc_t = accuracy_score(yt, pt)
    pre_t = precision_score(yt, pt, zero_division=0)
    rec_t = recall_score(yt, pt)
    f1_t  = f1_score(yt, pt)
    auc_t = roc_auc_score(yt, st)

    # 混淆矩阵
    cm_t = confusion_matrix(yt, pt)
    fig_cm, ax = plt.subplots(figsize=(4,3))
    sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f'{name} 测试集混淆矩阵')
    ax.set_xlabel('预测')
    ax.set_ylabel('真实')
    cm_path = out_dir / f'{name}_confusion.png'
    fig_cm.tight_layout(); fig_cm.savefig(cm_path, dpi=150); plt.close(fig_cm)

    # ROC
    fpr_v, tpr_v, _ = roc_curve(yv, sv)
    fpr_t, tpr_t, _ = roc_curve(yt, st)
    fig_roc, ax = plt.subplots(figsize=(4,3))
    ax.plot(fpr_v, tpr_v, label=f'验证 AUC={auc_v:.3f}')
    ax.plot(fpr_t, tpr_t, label=f'测试 AUC={auc_t:.3f}')
    ax.plot([0,1],[0,1],'k--',alpha=0.4)
    ax.set_title(f'{name} ROC曲线')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend()
    roc_path = out_dir / f'{name}_roc.png'
    fig_roc.tight_layout(); fig_roc.savefig(roc_path, dpi=150); plt.close(fig_roc)

    # PR 曲线
    pr_v, rc_v, _ = precision_recall_curve(yv, sv)
    pr_t, rc_t, _ = precision_recall_curve(yt, st)
    fig_pr, ax = plt.subplots(figsize=(4,3))
    ax.plot(rc_v, pr_v, label='验证')
    ax.plot(rc_t, pr_t, label='测试')
    ax.set_title(f'{name} PR曲线')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend()
    pr_path = out_dir / f'{name}_pr.png'
    fig_pr.tight_layout(); fig_pr.savefig(pr_path, dpi=150); plt.close(fig_pr)

    metrics = {
        'val': {'acc':acc_v,'pre':pre_v,'rec':rec_v,'f1':f1_v,'auc':auc_v},
        'test':{'acc':acc_t,'pre':pre_t,'rec':rec_t,'f1':f1_t,'auc':auc_t},
        'cm_path': str(cm_path), 'roc_path': str(roc_path), 'pr_path': str(pr_path)
    }
    return metrics

print('评估函数就绪，输出目录:', out_dir.resolve())


评估函数就绪，输出目录: C:\Users\31670\Desktop\Study-Material\MISC\实验二\outputs


In [14]:
# 13) 在验证/测试集评估并保存模型
best_svm = svm_gs.best_estimator_
best_dt  = dt_gs.best_estimator_

svm_metrics = eval_and_plots('SVM', best_svm, X_valid, y_valid, X_test, y_test)
dt_metrics  = eval_and_plots('DT',  best_dt,  X_valid, y_valid, X_test, y_test)

print('SVM 测试集指标:', svm_metrics['test'])
print('DT  测试集指标:', dt_metrics['test'])

joblib.dump(best_svm, out_dir / 'best_svm_pipeline.joblib')
joblib.dump(best_dt,  out_dir / 'best_dt_pipeline.joblib')

# 导出清洗后的特征表
clean_path = out_dir / 'clean_features.csv'
pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1).to_csv(clean_path, index=False)
print('已保存清洗数据到:', clean_path)


SVM 测试集指标: {'acc': 0.7133333333333334, 'pre': 0.515625, 'rec': 0.7333333333333333, 'f1': 0.6055045871559633, 'auc': 0.7868783068783068}
DT  测试集指标: {'acc': 0.6466666666666666, 'pre': 0.44285714285714284, 'rec': 0.6888888888888889, 'f1': 0.5391304347826087, 'auc': 0.7238095238095239}
已保存清洗数据到: outputs\clean_features.csv


In [15]:
# 14) 特征重要性与解释（DT、Permutation）
imp = None
if hasattr(best_dt.named_steps['clf'], 'feature_importances_'):
    # 获取One-Hot后的特征名
    ohe = best_dt.named_steps['pre'].named_transformers_['cat'].named_steps['onehot'] if len(cat_features) else None
    num_names = num_features
    cat_names = list(ohe.get_feature_names_out(cat_features)) if ohe is not None else []
    all_names = num_names + cat_names
    importances = best_dt.named_steps['clf'].feature_importances_
    order = np.argsort(importances)[::-1][:20]
    fig, ax = plt.subplots(figsize=(6,5))
    sns.barplot(x=importances[order], y=np.array(all_names)[order], ax=ax, orient='h')
    ax.set_title('DT 特征重要性Top20')
    fig.tight_layout(); fig.savefig(out_dir/'DT_feature_importance.png', dpi=150); plt.close(fig)
    imp = list(zip(np.array(all_names)[order].tolist(), importances[order].round(4).tolist()))

# 置换重要性（Permutation）
perm = permutation_importance(best_dt, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
fig, ax = plt.subplots(figsize=(6,5))
order = perm.importances_mean.argsort()[-20:]
ax.barh(np.array(range(len(order))), perm.importances_mean[order])
ax.set_yticks(np.array(range(len(order))))
ax.set_yticklabels(np.array(num_features + cat_features)[order])
ax.set_title('Permutation Importance (DT) Top20')
fig.tight_layout(); fig.savefig(out_dir/'DT_permutation_importance.png', dpi=150); plt.close(fig)

print('已生成特征重要性图。')


已生成特征重要性图。


In [16]:
# 15) 自动生成Word实验报告（插入指标与图）
report_path = out_dir / '实验二_心血管_实验报告.docx'
doc = Document()

doc.add_heading('实验二：心血管既往史数据清洗与SVM/决策树建模', 0)

doc.add_heading('一、数据概览', level=1)
doc.add_paragraph(f'主数据行列: {raw_main.shape}, 参考表: {None if raw_ref is None else raw_ref.shape}')
doc.add_paragraph('目标列：既往史-心血管（有=1，无=0）。本实验保留关键数值特征与常见行为特征，并对类别变量进行One-Hot编码。')


def metrics_table(doc, title, m):
    doc.add_heading(title, level=2)
    table = doc.add_table(rows=3, cols=6)
    hdr = table.rows[0].cells
    hdr[0].text = '数据集'; hdr[1].text = 'Accuracy'; hdr[2].text = 'Precision'; hdr[3].text = 'Recall'; hdr[4].text = 'F1'; hdr[5].text = 'ROC-AUC'
    vrow = table.rows[1].cells
    v = m['val']
    vrow[0].text = '验证'; vrow[1].text=f"{v['acc']:.3f}"; vrow[2].text=f"{v['pre']:.3f}"; vrow[3].text=f"{v['rec']:.3f}"; vrow[4].text=f"{v['f1']:.3f}"; vrow[5].text=f"{v['auc']:.3f}"
    trow = table.rows[2].cells
    t = m['test']
    trow[0].text = '测试'; trow[1].text=f"{t['acc']:.3f}"; trow[2].text=f"{t['pre']:.3f}"; trow[3].text=f"{t['rec']:.3f}"; trow[4].text=f"{t['f1']:.3f}"; trow[5].text=f"{t['auc']:.3f}"

metrics_table(doc, 'SVM 指标', svm_metrics)
metrics_table(doc, '决策树 指标', dt_metrics)

for title, m in [('SVM', svm_metrics), ('DT', dt_metrics)]:
    doc.add_heading(f'{title} 混淆矩阵与ROC/PR', level=2)
    for img_key in ['cm_path','roc_path','pr_path']:
        p = Path(m[img_key])
        if p.exists():
            doc.add_picture(str(p), width=Inches(4.8))

if (out_dir/'DT_feature_importance.png').exists():
    doc.add_heading('决策树特征重要性', level=2)
    doc.add_picture(str(out_dir/'DT_feature_importance.png'), width=Inches(5.5))
if (out_dir/'DT_permutation_importance.png').exists():
    doc.add_picture(str(out_dir/'DT_permutation_importance.png'), width=Inches(5.5))

# 公式说明
p = doc.add_paragraph('主要指标说明：')
p.add_run('F1').bold = True
p.add_run(' = 2 × (Precision × Recall) / (Precision + Recall)')

# 依赖版本
doc.add_heading('运行环境', level=1)
doc.add_paragraph(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | sklearn: {__import__('sklearn').__version__}")

doc.save(str(report_path))
print('已生成报告:', report_path.resolve())


已生成报告: C:\Users\31670\Desktop\Study-Material\MISC\实验二\outputs\实验二_心血管_实验报告.docx


# 16) 结论与后续工作（Markdown）
